In [23]:
import sys
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.saving import save_model
from game_board import GameBoard

In [24]:
version = tf.__version__
print("TensorFlow version:", version)
python_version = sys.version
print("Python version:", python_version)

TensorFlow version: 2.20.0
Python version: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]


In [25]:
def create_model(input_shape, num_actions):
    model = Sequential()
    model.add(Dense(64, activation='relu', input_shape=input_shape))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(num_actions, activation='linear'))
    return model

In [26]:

def apply_final_reward(episode, final_reward, K=5, gamma=0.9):
    n = len(episode)
    for i in range(n):
        s, a, r, ns, done, player, la, lna = episode[i]
        steps_from_end = n - 1 - i
        if steps_from_end < K:
            r += final_reward * (gamma ** steps_from_end)
        episode[i] = (s, a, r, ns, done, player, la, lna)
    return episode

In [27]:
class ReplayBuffer:
    def __init__(self, max_size=10000):
        self.buffer = []
        self.max_size = max_size

    def push(self, state, action, reward, next_state, done, legal_next_actions):
        if len(self.buffer) >= self.max_size:
            self.buffer.pop(0)
        self.buffer.append((state, action, reward, next_state, done, legal_next_actions))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

In [28]:
def train_step_on_batch(model, replay_buffer, batch_size=32, gamma=0.99):
    if len(replay_buffer) < batch_size:
        return
    batch = replay_buffer.sample(batch_size)
    states, actions, rewards, next_states, dones, legal_next_actions_list = zip(*batch)
    states = np.array(states)
    next_states = np.array(next_states)
    rewards = np.array(rewards, dtype=np.float32)
    dones = np.array(dones, dtype=np.float32)

    target_qs = model.predict(next_states, verbose=0)
    max_next_qs = np.array([
        np.max(target_qs[i][legal_next_actions_list[i]]) if legal_next_actions_list[i] else 0
        for i in range(len(batch))
    ])
    targets = rewards + (1 - dones) * gamma * max_next_qs

    q_values = model.predict(states, verbose=0)
    for i, action in enumerate(actions):
        q_values[i][action] = targets[i]

    model.train_on_batch(states, q_values)

In [29]:
def train_step_fit(model, replay_buffer, batch_size=32, gamma=0.99):
    if len(replay_buffer) < batch_size:
        return
    batch = replay_buffer.sample(batch_size)
    states, actions, rewards, next_states, dones, legal_next_actions_list = zip(*batch)
    states = np.array(states)
    next_states = np.array(next_states)
    rewards = np.array(rewards, dtype=np.float32)
    dones = np.array(dones, dtype=np.float32)

    target_qs = model.predict(next_states, verbose=0)
    max_next_qs = np.array([
        np.max(target_qs[i][legal_next_actions_list[i]]) if legal_next_actions_list[i] else 0
        for i in range(len(batch))
    ])
    targets = rewards + (1 - dones) * gamma * max_next_qs

    q_values = model.predict(states, verbose=0)
    for i, action in enumerate(actions):
        q_values[i][action] = targets[i]

    model.fit(states, q_values, epochs=1, verbose=0)

In [30]:
def play_against_model(board, model):
    board.reset()
    board.initialize()
    print("Game started!")
    while not board.game_finish:
        state = board.get_status()
        player = state[14]
        board.print_board()
        #print("Board state:", state)
        legal_actions = board.get_playable_pits()
        print("Playable moves:", legal_actions)
        if player == 1:
            while True:
                try:
                    user_action = int(input("Enter your move (pit index): "))
                    if user_action in legal_actions:
                        break
                    else:
                        print("Invalid move! Playable moves are:", legal_actions)
                except ValueError:
                    print("Please enter a valid number.")
            board.play(user_action, player)
            print(f"Your move: {user_action}")
        else:
            state_input = np.array(state).reshape(1, 16)
            q_values = model.predict(state_input, verbose=0)[0]
            best_action_index = np.argmax(q_values[legal_actions])
            action = legal_actions[best_action_index]
            board.play(action, player)
            print(f"Model's move: {action}")

    print("Game finished!")
    score_p1 = board.get_final_score(player=1)
    score_p2 = board.get_final_score(player=2)
    print(f"Your score: {score_p1}")
    print(f"Model's score: {score_p2}")
    if score_p1 > score_p2:
        print("You win!")
    elif score_p2 > score_p1:
        print("Model wins!")
    else:
        print("Draw!")

In [31]:
train_model = True
load_model = False
game_count=100000
board = GameBoard()
replay = ReplayBuffer(max_size=10000)
gamma = 0.99
K = 5
epsilon_start = 1.0
epsilon_end = 0.05
epsilon_decay_steps = 200

In [32]:
if load_model:
    model = tf.keras.models.load_model("models/mancala_model_20000.keras")
    print("Model loaded from file.")
else:
    model = create_model((22,), 14)
    model.compile(optimizer='adam',
    loss='mse',
    metrics=['accuracy'])
    model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 64)             │         1,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 14)             │           910 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,958 (74.05 KB)

 Trainable params: 18,958 (74.05 KB)

 Non-trainable params: 0 (0.00 B)

In [33]:
if train_model:
    for t in range(game_count):
        board.initialize()
        print(f"Game {t+1} started")
        episode_p0 = []
        episode_p1 = []

        while True:
            state = board.get_status()
            board.verbose = False
            print(state)
            player = state[14]
            state_input = np.array(state).reshape(1, 22)
            q_values = model.predict(state_input, verbose=0)[0]

            epsilon = max(epsilon_end, epsilon_start - (epsilon_start - epsilon_end) * t / epsilon_decay_steps) 
            legal_actions = board.get_playable_pits()
            if legal_actions:
                if random.random() < epsilon:
                    action = random.choice(legal_actions)
                else:
                    best_action_index = np.argmax(q_values[legal_actions])
                    action = legal_actions[best_action_index]
            else:
                break

            board.play(action, player)
            reward = board.get_score()
            next_state = board.get_status()
            next_state_input = np.array(next_state).reshape(1, 22)
            next_q_values = model.predict(next_state_input, verbose=0)[0]
            legal_next_actions = board.get_playable_pits()
            done = board.game_finish

            transition = (state, action, reward, next_state, done, player, legal_actions, legal_next_actions)
            if player == 0:
                episode_p0.append(transition)
            else:
                episode_p1.append(transition)
    
            if done:
                print(next_state)
                print("Game finished")
                score_p0 = board.get_final_score(player=0)
                score_p1 = board.get_final_score(player=1)
                if score_p0 > score_p1:
                    final_reward_p0 = 1
                    final_reward_p1 = -1
                elif score_p0 < score_p1:
                    final_reward_p0 = -1
                    final_reward_p1 = 1
                else:
                    final_reward_p0 = 0
                    final_reward_p1 = 0
    
                episode_p0 = apply_final_reward(episode_p0, final_reward_p0, K=K, gamma=gamma)
                episode_p1 = apply_final_reward(episode_p1, final_reward_p1, K=K, gamma=gamma)
    
                for transition in episode_p0 + episode_p1:
                    s, a, r, ns, done, player, la, lna = transition
                    replay.push(s, a, r, ns, done, lna)
    
                board.reset()
                break
    
        if (t + 1) % 1000 == 0:
            save_model(model, f'models/mancala_model_{t+1}.keras')
            print(f"Model saved at iteration {t+1}")

Game 1 started
[4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0, 1, 0, 0, 24, 24, 0, 0, 1]
[4, 0, 5, 5, 5, 5, 0, 4, 4, 4, 4, 4, 4, 0, 2, 0, 0, 24, 24, 0, 1, 1]
[4, 0, 5, 5, 5, 5, 0, 4, 0, 5, 5, 5, 5, 0, 1, 0, 0, 24, 24, 1, 1, 1]
[4, 0, 5, 5, 0, 6, 7, 5, 1, 0, 5, 5, 5, 0, 2, 0, -7, 21, 20, 1, 2, 1]
[5, 1, 0, 5, 0, 6, 7, 5, 1, 0, 5, 0, 6, 7, 1, 0, 0, 17, 17, 2, 2, 1]
[5, 0, 0, 5, 0, 6, 13, 5, 1, 0, 0, 0, 6, 7, 2, 0, -6, 12, 16, 3, 3, 1]
[5, 0, 0, 0, 0, 6, 13, 5, 0, 0, 0, 0, 6, 13, 1, 0, 0, 11, 11, 4, 4, 1]
[5, 0, 0, 0, 0, 0, 14, 6, 1, 1, 1, 1, 6, 13, 2, 0, -1, 16, 5, 0, 5, 1]
[5, 0, 0, 0, 0, 0, 14, 0, 2, 2, 2, 2, 7, 14, 2, 0, 0, 15, 5, 1, 5, 1]
[5, 0, 0, 0, 0, 0, 14, 0, 2, 0, 3, 3, 7, 14, 1, 0, 0, 5, 15, 5, 2, 0]
[0, 1, 1, 1, 1, 1, 14, 0, 2, 0, 3, 3, 7, 14, 2, 0, 0, 15, 5, 2, 1, 1]
[0, 1, 1, 1, 1, 1, 14, 0, 0, 1, 4, 3, 7, 14, 1, 0, 0, 5, 15, 1, 2, 0]
[0, 1, 1, 1, 0, 2, 14, 0, 0, 1, 4, 3, 7, 14, 2, 0, 0, 15, 5, 2, 2, 1]
[1, 2, 2, 2, 1, 3, 14, 0, 0, 1, 4, 3, 0, 15, 1, 0, -1, 11, 8, 0, 3, 2]
[1, 2

KeyboardInterrupt: 

In [ ]:
if train_model:
    for i in range(game_count // 100):
        train_step_fit(model, replay, batch_size=32, gamma=gamma)
        train_step_on_batch(model, replay, batch_size=32, gamma=gamma)

In [ ]:
save_model(model, 'mancala_model.keras')

In [ ]:
# play_against_model(board, model)